In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

In [8]:
from pydantic import Field, BaseModel
from langchain_core.prompts import ChatPromptTemplate

class Workout(BaseModel):
    workoutName: str = Field(description="The exact name of the gym/fitness exercise")
    workoutMuscleTarget: list[str] = Field(
        description=(
            "List of specific muscle names (anatomical) that are PRIMARY and SECONDARY targets. "
            "Use precise muscle names e.g. 'Pectoralis Major', 'Triceps Brachii', 'Anterior Deltoid'. "
            "Order by most to least targeted."
        )
    )
    workoutType: str = Field(description="Category: 'Compound' or 'Isolation'")

system_prompt = """You are a certified fitness expert and exercise physiologist.
When given an exercise/workout name, identify ALL muscles targeted — both primary movers and secondary stabilizers.
Always use proper anatomical muscle names.
Be comprehensive and accurate based on exercise science."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "List the muscles targeted by this exercise: {workout}")
])

structured_llm = llm.with_structured_output(Workout)
chain = prompt | structured_llm


In [9]:
result = chain.invoke({"workout": "Bench Press"})

print(f"Exercise: {result.workoutName}")
print(f"Type:     {result.workoutType}")
print("Muscles targeted:")
for i, muscle in enumerate(result.workoutMuscleTarget, 1):
    print(f"  {i}. {muscle}")


Exercise: Bench Press
Type:     Compound
Muscles targeted:
  1. Pectoralis Major
  2. Anterior Deltoid
  3. Serratus Anterior
  4. Trapezius
  5. Rhomboids
  6. Triceps Brachii


In [14]:
from typing import TypedDict, Annotated
from langchain_core.prompts import ChatPromptTemplate

class WorkoutDict(TypedDict):
    """Information about a gym workout exercise including name, targeted muscles, and exercise type."""
    workoutName: Annotated[str, "The exact name of the gym/fitness exercise"]
    workoutMuscleTarget: Annotated[
        list[str], 
        "List of specific muscle names (anatomical) that are PRIMARY and SECONDARY targets. Order by most to least targeted."
    ]
    workoutType: Annotated[str, "Category: 'Compound' or 'Isolation'"]

system_prompt = """You are a certified fitness expert and exercise physiologist.
When given an exercise/workout name, identify ALL muscles targeted — both primary movers and secondary stabilizers.
Always fill out all required fields: workoutName, workoutMuscleTarget, and workoutType.
Always use proper anatomical muscle names.
Be comprehensive and accurate based on exercise science."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "List the muscles targeted by this exercise: {workout}")
])

structured_llm = llm.with_structured_output(WorkoutDict)
chain = prompt | structured_llm


In [15]:
result = chain.invoke({"workout": "Bench Press"})

# Note: Since it's a dict now, access values using dictionary key syntax result["key"]
print(f"Exercise: {result['workoutName']}")
print(f"Type:     {result['workoutType']}")
print("Muscles targeted:")
for i, muscle in enumerate(result['workoutMuscleTarget'], 1):
    print(f"  {i}. {muscle}")


Exercise: Bench Press
Type:     Compound
Muscles targeted:
  1. Pectoralis major (anterior head)
  2. Anterior deltoid
  3. Triceps brachii (long head)


In [22]:
from dataclasses import dataclass, field
from langchain_core.prompts import ChatPromptTemplate

@dataclass
class WorkoutDataClass:
    """Information about a gym workout exercise including name, targeted muscles, and exercise type."""
    workoutName: str = field(
        metadata={"description": "The exact name of the gym/fitness exercise"}
    )
    workoutMuscleTarget: list[str] = field(
        metadata={"description": "List of specific muscle names (anatomical) that are PRIMARY and SECONDARY targets. Order by most to least targeted."}
    )
    workoutType: str = field(
        metadata={"description": "Category: 'Compound' or 'Isolation'"}
    )

system_prompt = """
You are a certified fitness expert and exercise physiologist.
When given an exercise/workout name, identify ALL muscles targeted — both primary movers and secondary stabilizers.
Always fill out all required fields: workoutName, workoutMuscleTarget, and workoutType.
Always use proper anatomical muscle names.
Be comprehensive and accurate based on exercise science.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "List the muscles targeted by this exercise: {workout}")
])

# Pass Dataclass directly to with_structured_output
structured_llm = llm.with_structured_output(WorkoutDataClass)
chain = prompt | structured_llm


In [ ]:
data_dict = chain.invoke({"workout": "Bench Press"})

result = WorkoutDataClass(**data_dict)

# Since result is an instance of WorkoutDataClass, access properties using dot notation (.)
print(result)
print(f"Exercise: {result.workoutName}")
print(f"Type:     {result.workoutType}")
print("Muscles targeted:")
for i, muscle in enumerate(result.workoutMuscleTarget, 1):
    print(f"  {i}. {muscle}")


WorkoutDataClass(workoutName='Bench Press', workoutMuscleTarget=['Pectoralis major', 'Anterior deltoids', 'Triceps brachii', 'Serratus anterior', 'Trapezius', 'Rhomboids', 'Scapular stabilizers'], workoutType='Resistance training')
Exercise: Bench Press
Type:     Resistance training
Muscles targeted:
  1. Pectoralis major
  2. Anterior deltoids
  3. Triceps brachii
  4. Serratus anterior
  5. Trapezius
  6. Rhomboids
  7. Scapular stabilizers


: 